In [13]:
from web3 import Web3
import json

In [14]:
# Load config
with open("config.json") as f:
    config = json.load(f)

w3 = Web3(Web3.HTTPProvider(config["blockchain"]["rpc_url"]))
print("Connected:", w3.is_connected())

stablecoin_address = config["blockchain"]["stablecoin_address"]
p2p_market_address = config["blockchain"]["p2p_market_address"]

# Minimal ERC-20 ABI — only what we need for checking allowances
erc20_abi = [
    {
        "constant": True,
        "inputs": [
            {"name": "owner", "type": "address"},
            {"name": "spender", "type": "address"}
        ],
        "name": "allowance",
        "outputs": [{"name": "", "type": "uint256"}],
        "type": "function"
    },
    {
        "constant": True,
        "inputs": [{"name": "account", "type": "address"}],
        "name": "balanceOf",
        "outputs": [{"name": "", "type": "uint256"}],
        "type": "function"
    }
]

stablecoin = w3.eth.contract(address=Web3.to_checksum_address(stablecoin_address), abi=erc20_abi)

Connected: True


In [22]:
account_Deploy = w3.eth.account.from_key(os.environ["DEPLOYER_PRIVATE_KEY"])
print("Connected, chain id:", w3.eth.chain_id)
print("Account:", account_Deploy.address)
print("Balance (ETH):", w3.from_wei(w3.eth.get_balance(account_Deploy.address), "ether"))

Connected, chain id: 11155111
Account: 0x2C4b47689a05f0653637a97a8Db762b764f1653c
Balance (ETH): 0.385911389090511201


In [23]:
account_trigger = w3.eth.account.from_key(os.environ["TRIGGER_PRIVATE_KEY"])
print("Connected, chain id:", w3.eth.chain_id)
print("Account:", account_trigger.address)
print("Balance (ETH):", w3.from_wei(w3.eth.get_balance(account_trigger.address), "ether"))

Connected, chain id: 11155111
Account: 0x4022d2250AB1E76d3fcbCcf18d39656f4559b83c
Balance (ETH): 0.175581917347971325


In [26]:
def load_contract(abi_path: str, address: str):
    with open(abi_path) as f:
        artifact = json.load(f)
    abi = artifact["abi"] if "abi" in artifact else artifact
    return w3.eth.contract(address=Web3.to_checksum_address(address), abi=abi)

bc = config["blockchain"]

oracle_storage = load_contract("abi/OracleStorage.json", bc["oracle_storage_address"])
p2p_market = load_contract("abi/P2PEnergyMarket.json", bc["p2p_market_address"])

oracle_storage.address, p2p_market.address

('0x61769c1C299495194D7Df49030019e613d13a88D',
 '0xB5D7DE4841985feA6a234256B5Adfa4AF1725bF7')

In [27]:
for household in config["households"]:
    addr = Web3.to_checksum_address(household["address"])
    allowance = stablecoin.functions.allowance(addr, p2p_market_address).call()
    balance = stablecoin.functions.balanceOf(addr).call()
    print(f"{household.get('name', addr)}: allowance={allowance}, balance={balance}")

0x4022d2250AB1E76d3fcbCcf18d39656f4559b83c: allowance=50, balance=100000000
0xF49153d700AD86CA224f7B2064F541278FE1c320: allowance=50, balance=5000000


In [31]:
oracle_storage.functions.authorizeOracle(os.getenv("ORACLE_PRIVATE_KEY"))

<Function authorizeOracle(address) bound to ('bb0a0d5667ab20e5f60c80e65afef6188f90a92cf72e689aca335b2bf9dc57b7',)>

In [ ]:
for household in config["households"]:
    p2p_market.functions.registerHousehold(household["address"])

In [ ]:
p2p Market Registierung
Oracel writer registeirt households automatisch für Oracle Storage. Wir müssen dasselbe für die p2p market regiestrierung aufbauen.


In [33]:
# Registierung P2P Market
def register_households_if_needed(self):
        """Stellt sicher, dass alle konfigurierten Haushalte im Oracle registriert sind."""
        for h in self.config["households"]:
            addr = Web3.to_checksum_address(h["address"])
            # registered = self.p2p_market.functions.isHouseholdRegistered(addr).call()
            registered = False
            if not registered:
                print(f"Registriere Haushalt {h['id']} ({addr}) ...")
                self._send_tx(self.p2p_market.functions.registerHousehold(addr))

In [ ]:
for h in self.config["households"]:
    addr = Web3.to_checksum_address(h["address"])
    # registered = self.p2p_market.functions.isHouseholdRegistered(addr).call()
    print(f"Registriere Haushalt {h['id']} ({addr}) ...")
    self._send_tx(self.p2p_market.functions.registerHousehold(addr))

TypeError: register_households_if_needed() missing 1 required positional argument: 'self'

In [ ]:
COntract keien Whitelist
p2p gibt Teilnehmern direkt das Geld.
Swiss stablecoin nicht richtig gewhitelisted. 


In [ ]:
House2 versucht zu schreiben
